# Building the NYC Parking Analytics SQLite Database

This notebook documents how the cleaned parking data, daily weather data, violation fine lookup, and Census population data are organized into a relational SQLite database.

The database is created with Python so the process is repeatable. The large database file is not stored in Git because it can be rebuilt from the project scripts and source files.

## Why use a relational database?

The cleaned parking file contains millions of ticket records. Weather, violation fines, and Census population describe groups of those tickets rather than individual tickets. Keeping those sources in separate tables avoids repeating the same information millions of times.

The design uses:

- `parking_violations`: one row per ticket
- `weather_daily`: one row per date
- `violation_lookup`: one row per violation code
- `census_borough`: one row per NYC borough/county
- `source_metadata`: source links and descriptions

## Entity Relationship Diagram

An Entity Relationship Diagram, or ERD, shows the tables, their keys, and how the tables relate to each other.

```text
weather_daily.weather_date       1 ---- many parking_violations.issue_date
violation_lookup.violation_code  1 ---- many parking_violations.violation_code
census_borough.borough           1 ---- many parking_violations.borough
```

A complete ERD is also stored in `reports/ERD.md`.

In [1]:
from pathlib import Path
import sqlite3
import sys

import pandas as pd

## Locate the project files

The path check allows the notebook to work whether VS Code starts it from the project root or from the `notebooks` folder.

In [2]:
STARTING_DIR = Path.cwd()
PROJECT_ROOT = STARTING_DIR

if not (PROJECT_ROOT / "src" / "nycparking").exists():
    PROJECT_ROOT = STARTING_DIR.parent

SRC_DIR = PROJECT_ROOT / "src"
DATABASE_FILE = PROJECT_ROOT / "data" / "database" / "nyc_parking.sqlite"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Database: {DATABASE_FILE}")

Project root: c:\Users\jayson.coker\Documents\nyc-parking-analytics
Database: c:\Users\jayson.coker\Documents\nyc-parking-analytics\data\database\nyc_parking.sqlite


## Build the database

The `build_database` function performs the complete process:

1. Creates a new SQLite database and relational schema.
2. Loads weather data.
3. Cleans and loads the violation fine lookup.
4. Downloads and caches the five NYC county Census records.
5. Loads the parking file in chunks so memory use stays manageable.
6. Creates indexes after the large load finishes.
7. Checks row counts, joins, and foreign keys.

In [3]:
from nycparking.sqlite.build_database import build_database

# This rebuilds the database and may take several minutes.
# build_results = build_database(DATABASE_FILE)
# build_results

The build cell is commented out so the database is not accidentally rebuilt each time the notebook opens. Uncomment it when the source data changes or the database needs to be recreated.

## Connect to SQLite and review the tables

In [4]:
connection = sqlite3.connect(DATABASE_FILE)
connection.execute("PRAGMA foreign_keys = ON")

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    connection,
)
tables

,name
0,census_borough
1,parking_violations
2,source_metadata
3,violation_lookup
4,weather_daily


## Validate table row counts

The parking table should match the cleaned CSV row count. The dimension tables should have one row per date, violation code, or borough.

In [5]:
row_counts = pd.read_sql_query(
    """
    SELECT 'parking_violations' AS table_name, COUNT(*) AS row_count FROM parking_violations
    UNION ALL
    SELECT 'weather_daily', COUNT(*) FROM weather_daily
    UNION ALL
    SELECT 'violation_lookup', COUNT(*) FROM violation_lookup
    UNION ALL
    SELECT 'census_borough', COUNT(*) FROM census_borough
    UNION ALL
    SELECT 'source_metadata', COUNT(*) FROM source_metadata
    """,
    connection,
)
row_counts

,table_name,row_count
0,parking_violations,7056788
1,weather_daily,9486
2,violation_lookup,99
3,census_borough,5
4,source_metadata,4


## Validate the relationships

These checks look for parking rows that fail to find a matching weather date, violation code, or borough Census row. A result of zero means the relationship is complete.

In [6]:
relationship_checks = pd.read_sql_query(
    """
    SELECT 'Missing weather match' AS check_name, COUNT(*) AS problem_rows
    FROM parking_violations AS p
    LEFT JOIN weather_daily AS w ON w.weather_date = p.issue_date
    WHERE w.weather_date IS NULL

    UNION ALL

    SELECT 'Missing violation lookup', COUNT(*)
    FROM parking_violations AS p
    LEFT JOIN violation_lookup AS v ON v.violation_code = p.violation_code
    WHERE p.violation_code IS NOT NULL AND v.violation_code IS NULL

    UNION ALL

    SELECT 'Missing Census match', COUNT(*)
    FROM parking_violations AS p
    LEFT JOIN census_borough AS c ON c.borough = p.borough
    WHERE p.borough IS NOT NULL AND c.borough IS NULL
    """,
    connection,
)
relationship_checks

,check_name,problem_rows
0,Missing weather match,0
1,Missing violation lookup,0
2,Missing Census match,0


## SQL query 1: Weather and average daily ticket volume

This query first counts tickets for each date in a subquery. It then joins those daily totals to weather and compares average ticket volume by weather condition. The `HAVING` clause removes conditions with too few active days to support a useful comparison.

In [7]:
weather_analysis = pd.read_sql_query(
    """
    SELECT
        w.weather_condition,
        COUNT(*) AS active_days,
        ROUND(AVG(d.ticket_count), 2) AS average_daily_tickets,
        ROUND(AVG(w.precipitation), 2) AS average_precipitation
    FROM (
        SELECT issue_date, COUNT(*) AS ticket_count
        FROM parking_violations
        GROUP BY issue_date
    ) AS d
    JOIN weather_daily AS w ON w.weather_date = d.issue_date
    GROUP BY w.weather_condition
    HAVING COUNT(*) >= 5
    ORDER BY average_daily_tickets DESC
    """,
    connection,
)
weather_analysis

,weather_condition,active_days,average_daily_tickets,average_precipitation
0,Clear sky,30,17957.50,0.00
1,Partly cloudy,34,13423.59,0.00
2,Light drizzle,103,11012.91,0.36
3,Heavy rain,27,10978.00,28.45
4,Overcast,262,10762.77,0.00
5,Dense drizzle,19,10299.58,3.34
6,Mainly clear,18,10033.72,0.00
7,Slight rain,43,9334.44,5.06
8,Moderate rain,77,7422.25,14.46
9,Moderate drizzle,59,7251.86,2.02


## SQL query 2: Estimated fine exposure by violation type

This query joins tickets to the supplied fine lookup, groups by violation code, and estimates total listed fine exposure. Violations marked as variable or missing are excluded from the dollar calculation. This is not actual collected revenue because payments, penalties, reductions, and program rules are not included.

In [8]:
fine_analysis = pd.read_sql_query(
    """
    SELECT
        p.violation_code,
        v.violation_description,
        COUNT(*) AS ticket_count,
        v.fine_amount,
        ROUND(COUNT(*) * v.fine_amount, 2) AS estimated_fine_exposure
    FROM parking_violations AS p
    JOIN violation_lookup AS v ON v.violation_code = p.violation_code
    WHERE v.fine_amount IS NOT NULL
    GROUP BY p.violation_code, v.violation_description, v.fine_amount
    HAVING COUNT(*) >= 100
    ORDER BY estimated_fine_exposure DESC
    LIMIT 20
    """,
    connection,
)
fine_analysis

,violation_code,violation_description,ticket_count,fine_amount,estimated_fine_exposure
0,36,PHTO SCHOOL ZN SPEED VIOLATION,2311451,50.0,115572550.0
1,21,NO PARKING-STREET CLEANING,795556,65.0,51711140.0
2,14,NO STANDING-DAY/TIME LIMITS,349737,115.0,40219755.0
3,40,FIRE HYDRANT,263024,115.0,30247760.0
4,38,FAIL TO DSPLY MUNI METER RECPT,520865,35.0,18230275.0
5,5,BUS LANE VIOLATION,353589,50.0,17679450.0
6,46,DOUBLE PARKING,144815,115.0,16653725.0
7,7,FAILURE TO STOP AT RED LIGHT,318581,50.0,15929050.0
8,20,NO PARKING-DAY/TIME LIMITS,259355,60.0,15561300.0
9,71,INSP. STICKER-EXPIRED/MISSING,229858,65.0,14940770.0


## SQL query 3: Parking tickets per 1,000 residents

Raw ticket counts can make larger boroughs look more active simply because they have more residents. This query joins ticket totals to Census population and calculates a rate per 1,000 residents.

In [9]:
borough_analysis = pd.read_sql_query(
    """
    SELECT
        c.borough,
        c.population,
        COUNT(p.summons_number) AS ticket_count,
        ROUND(COUNT(p.summons_number) * 1000.0 / c.population, 2) AS tickets_per_1000_residents
    FROM census_borough AS c
    LEFT JOIN parking_violations AS p ON p.borough = c.borough
    GROUP BY c.borough, c.population
    ORDER BY tickets_per_1000_residents DESC
    """,
    connection,
)
borough_analysis

,borough,population,ticket_count,tickets_per_1000_residents
0,Manhattan,1664862,1857772,1115.87
1,Queens,2358182,1992488,844.93
2,Bronx,1406332,968981,689.01
3,Brooklyn,2653963,1779380,670.46
4,Staten Island,501290,300389,599.23


## Database conclusions

The SQLite database separates individual ticket records from descriptive dimensions. This makes the relationships easier to understand, prevents repeated fine and population values, and supports SQL joins that answer the project questions.

The next step is exploratory analysis and visualization using the SQL query results.

In [10]:
connection.close()